# Finding the Greater-Than Circuit Using EAP(-IG)

First, we import various packages.

In [1]:
import os
# Run any op that Apple-silicon GPUs (MPS) do not support on the CPU instead of erroring
os.environ.setdefault('PYTORCH_ENABLE_MPS_FALLBACK', '1')

from functools import partial

import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import PreTrainedTokenizer
from transformer_lens import HookedTransformer

from eap.graph import Graph
from eap.evaluate import evaluate_graph, evaluate_baseline
from eap.attribute import attribute 

/opt/anaconda3/envs/ioi/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Dataset and Metrics

This package expects data to come from a dataloader. Each item consists of clean and corrupted paired inputs (strings), as well as a label (encoded as a token id). For convenience, we've included a dataset in that form as a CSV (more to come with the full code of the paper).

A metric takes in the model's (possibly corrupted) logits, clean logits, input lengths, and labels. It computes a metric value for each batch item; this can either be used as is, or turned into a loss (lower is better), or meaned.

In [2]:
def collate_EAP(xs):
    clean, corrupted, labels = zip(*xs)
    clean = list(clean)
    corrupted = list(corrupted)
    return clean, corrupted, labels

class EAPDataset(Dataset):
    def __init__(self, filepath):
        self.df = pd.read_csv(filepath)

    def __len__(self):
        return len(self.df)
    
    def shuffle(self):
        self.df = self.df.sample(frac=1)

    def head(self, n: int):
        self.df = self.df.head(n)
    
    def __getitem__(self, index):
        row = self.df.iloc[index]
        return row['clean'], row['corrupted'], row['label']
    
    def to_dataloader(self, batch_size: int):
        return DataLoader(self, batch_size=batch_size, collate_fn=collate_EAP)
    
def get_logit_positions(logits: torch.Tensor, input_length: torch.Tensor):
    batch_size = logits.size(0)
    idx = torch.arange(batch_size, device=logits.device)

    logits = logits[idx, input_length - 1]
    return logits

def get_prob_diff(tokenizer: PreTrainedTokenizer):
    year_indices = torch.tensor([tokenizer(f'{year:02d}').input_ids[0] for year in range(100)])

    def prob_diff(logits: torch.Tensor, clean_logits: torch.Tensor, input_length: torch.Tensor, labels: torch.Tensor, mean=True, loss=False):
        logits = get_logit_positions(logits, input_length)
        probs = torch.softmax(logits, dim=-1)[:, year_indices.to(logits.device)]

        results = []
        for prob, year in zip(probs, labels):
            results.append(prob[year + 1 :].sum() - prob[: year + 1].sum())
    
        results = torch.stack(results)
        if loss:
            results = -results
        if mean: 
            results = results.mean()
        return results
    return prob_diff

def kl_div(logits: torch.Tensor, clean_logits: torch.Tensor, input_length: torch.Tensor, labels: torch.Tensor, mean=True, loss=True):
    logits = get_logit_positions(logits, input_length)
    clean_logits = get_logit_positions(clean_logits, input_length)

    probs = torch.softmax(logits, dim=-1)
    clean_probs = torch.softmax(clean_logits, dim=-1)

    results = kl_div(probs.log(), clean_probs.log(), log_target=True, reduction='none').mean(-1)
    return results.mean() if mean else results


## Performing EAP-IG

First, we load the model, data, and metric.

In [3]:
model_name = 'gpt2-small'
if torch.cuda.is_available():
    device = 'cuda'
elif torch.backends.mps.is_available():
    device = 'mps'
else:
    device = 'cpu'
print(f'Using device: {device}')

model = HookedTransformer.from_pretrained(model_name, device=device)
model.cfg.use_split_qkv_input = True
model.cfg.use_attn_result = True
model.cfg.use_hook_mlp_in = True

ds = EAPDataset('greater_than_data.csv')
# 120 fits on a large CUDA GPU; smaller batches keep MPS / CPU within memory
BATCH_SIZE = 120 if device == 'cuda' else 20
dataloader = ds.to_dataloader(BATCH_SIZE)
prob_diff = get_prob_diff(model.tokenizer)

Using device: mps


/opt/anaconda3/envs/ioi/lib/python3.11/site-packages/transformer_lens/HookedTransformerConfig.py:343: UserWarning: MPS backend may produce silently incorrect results (PyTorch 2.14.0). Set TRANSFORMERLENS_ALLOW_MPS=1 to suppress this warning. See: https://github.com/TransformerLensOrg/TransformerLens/issues/1178
  utils.warn_if_mps(self.device)
`torch_dtype` is deprecated! Use `dtype` instead!


Loaded pretrained model gpt2-small into HookedTransformer


Then, we perform EAP! We instantiate an unscored graph from the model, and use the attribute method to score it. This requires a model, graph, dataloader, and loss. We set `method='EAP-IG'`, and set the number of iterations via `ig_steps`.

In [4]:
# Instantiate a graph with a model
g = Graph.from_model(model)

# Attribute using the model, graph, clean / corrupted data and labels, as well as a metric
attribute(model, g, dataloader, partial(prob_diff, loss=True, mean=True), method='EAP-IG-inputs', ig_steps=5)

100%|██████████| 50/50 [39:41<00:00, 47.63s/it]


We can now apply greedy search to the scored graph to find a circuit! We prune dead nodes, and export the circuit.

In [5]:
g.apply_topn(200, True)
g.to_json('graph.json')

We can then convert our circuit into a visualization!

In [6]:
try:
    import pygraphviz
    gz = g.to_image(f'graph.png')
except ImportError:
    print("No pygraphviz installed; skipping this part")

No pygraphviz installed; skipping this part


We then evaluate our model's metric score as opposed to a baseline.

In [7]:
baseline = evaluate_baseline(model, dataloader, partial(prob_diff, loss=False, mean=False)).mean().item()
results = evaluate_graph(model, g, dataloader, partial(prob_diff, loss=False, mean=False)).mean().item()
print(f"Original performance was {baseline}; the circuit's performance is {results}")


corrupted_baseline = evaluate_baseline(model, dataloader, partial(prob_diff, loss=False, mean=False), run_corrupted=True).mean().item()
faithfulness = (results - corrupted_baseline) / (baseline - corrupted_baseline)
print(f"Corrupted baseline: {corrupted_baseline}; normalized faithfulness: {faithfulness}")

100%|██████████| 50/50 [03:55<00:00,  4.70s/it]


Original performance was 0.8143222332000732; the circuit's performance is 0.7449120879173279


100%|██████████| 50/50 [03:31<00:00,  4.24s/it]

Corrupted baseline: -0.4565723240375519; normalized faithfulness: 0.9453848119126319


In [8]:
g.count_included_nodes(), g.count_included_edges()

(29, 150)

We can now compare that to a circuit found with vanilla EAP.

In [9]:
# Instantiate a graph with a model
g_eap = Graph.from_model(model)

# Attribute using the model, graph, clean / corrupted data and labels, as well as a metric
attribute(model, g_eap, dataloader, partial(prob_diff, loss=True, mean=True), method='EAP')

g_eap.apply_topn(200, True)

results_eap = evaluate_graph(model, g_eap, dataloader, partial(prob_diff, loss=False, mean=False)).mean().item()
print(f"Original performance was {baseline}; the circuit's performance is {results_eap}")

faithfulness_eap = (results_eap - corrupted_baseline) / (baseline - corrupted_baseline)
print(f"EAP normalized faithfulness: {faithfulness_eap}")


100%|██████████| 50/50 [04:04<00:00,  4.90s/it]

Original performance was 0.8143222332000732; the circuit's performance is 0.6563277244567871
EAP normalized faithfulness: 0.8756824412823847


We can also test other EAP-IG variants:

In [10]:
# Instantiate a graph with a model
g_cc = Graph.from_model(model)

# Attribute using the model, graph, clean / corrupted data and labels, as well as a metric
attribute(model, g_cc, dataloader, partial(prob_diff, loss=True, mean=True), method='clean-corrupted')

g_cc.apply_topn(200, True)

results_cc = evaluate_graph(model, g_cc, dataloader, partial(prob_diff, loss=False, mean=False)).mean().item()
print(f"Original performance was {baseline}; the circuit's performance is {results_cc}")

faithfulness_cc = (results_cc - corrupted_baseline) / (baseline - corrupted_baseline)
print(f"Clean-corrupted normalized faithfulness: {faithfulness_cc}")


100%|██████████| 50/50 [04:24<00:00,  5.29s/it]

Original performance was 0.8143222332000732; the circuit's performance is 0.698186993598938
Clean-corrupted normalized faithfulness: 0.908619295802507
